In [ ]:
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk

import polars as pl
import spacy

from nltk.corpus import stopwords
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer, MultiLabelBinarizer, normalize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    classification_report, hamming_loss, coverage_error,
    f1_score, roc_curve, auc,
    confusion_matrix, accuracy_score, jaccard_score
)


from sklearn.tree import DecisionTreeClassifier
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('outputs/plots',  exist_ok=True)

nltk.download('stopwords', quiet=True)

In [ ]:
df_train = pl.read_parquet('../../../EDA/df_train.parquet')
df_test = pl.read_parquet('../../../EDA/df_test.parquet')
df_val = pl.read_parquet('../../../EDA/df_val.parquet')

x_train_raw = df_train.select('text').to_series().to_list()
x_val_raw   = df_val.select('text').to_series().to_list()
x_test_raw  = df_test.select('text').to_series().to_list()

In [ ]:
!python -m spacy download en_core_web_lg


In [ ]:
nlp = spacy.load('en_core_web_lg', disable=['parser', 'ner'])

def decontract(sentence):
    sentence = re.sub(r"n\'t", ' not', sentence)
    sentence = re.sub(r"\'re",  ' are',   sentence)
    sentence = re.sub(r"\'s",   ' is',    sentence)
    sentence = re.sub(r"\'d",   ' would', sentence)
    sentence = re.sub(r"\'ll",  ' will',  sentence)
    sentence = re.sub(r"\'t",   ' not',   sentence)
    sentence = re.sub(r"\'ve",  ' have',  sentence)
    sentence = re.sub(r"\'m",   ' am',    sentence)
    return sentence

def removePunctuation(sentence):
    sentence = re.sub(r"[?|!'\"#]", '', sentence)
    sentence = re.sub(r'[.|,|)|(|\\|/]', ' ', sentence)
    return sentence.strip().replace('\n', ' ')

def removeNumber(sentence):
    return ' '.join(re.sub('[^a-z A-Z]+', '', w) for w in sentence.split()).strip()

def removeStopWords(sentence):
    stop_words = set(stopwords.words('english'))
    return ' '.join(w for w in sentence.split() if w.lower() not in stop_words)

def custom_preprocess(text):
    text = decontract(text)
    text = removePunctuation(text)
    text = removeNumber(text)
    text = removeStopWords(text)
    doc  = nlp(text)
    return ' '.join(token.lemma_ for token in doc if token.lemma_ != '-PRON-')

def spacy_embeddings(texts):
    texts_proc = [decontract(t) for t in texts]
    docs    = list(nlp.pipe(texts_proc, batch_size=50))
    vectors = np.array([doc.vector for doc in docs])
    return normalize(vectors, norm='l2')

In [ ]:
feature_union = FeatureUnion([
    ('tfidf', TfidfVectorizer(
        preprocessor=custom_preprocess,
        ngram_range=(1, 2),
        max_features=8000,
        norm='l2'
    )),
    ('embeddings', FunctionTransformer(
        spacy_embeddings,
        validate=False
    ))
])

print('Fitting feature union...')
X_train = feature_union.fit_transform(x_train_raw)
X_val   = feature_union.transform(x_val_raw)
X_test  = feature_union.transform(x_test_raw)
print(f'Shapes — train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}')

In [ ]:
y_train_raw = df_train['labels'].to_list()
y_val_raw   = df_val['labels'].to_list()
y_test_raw  = df_test['labels'].to_list()

all_labels_combined  = y_train_raw + y_val_raw + y_test_raw
all_unique_label_ids = sorted(set(item for sub in all_labels_combined for item in sub))

mlb = MultiLabelBinarizer(classes=all_unique_label_ids)
mlb.fit(all_labels_combined)

y_train_multilabel = mlb.transform(y_train_raw)
y_val_multilabel   = mlb.transform(y_val_raw)
y_test_multilabel  = mlb.transform(y_test_raw)

label_dict  = {0:'joy', 1:'trust', 2:'fear', 3:'surprise',
               4:'sadness', 5:'neutral', 6:'anticipation', 7:'anger', 8:'disgust'}
label_names = [label_dict[c] for c in mlb.classes_]
n_classes   = y_test_multilabel.shape[1]
print('Classes:', label_names)

In [ ]:
def plot_roc(y_true, y_score, title, save_path):
    sns.set_theme(style='ticks')
    plt.figure(figsize=(10, 6))
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_score[:, i])
        roc_auc     = auc(fpr, tpr)
        name        = label_dict[mlb.classes_[i]].title()
        plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guess')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right', fontsize='small')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=400)
    plt.show()
    print(f'Saved → {save_path}')


def plot_confusion_matrices(y_true, y_pred, title, save_path):
    ncols = 3
    nrows = (n_classes + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
    axes = axes.flatten()
    for i, ax in enumerate(axes[:n_classes]):
        cm = confusion_matrix(y_true[:, i], y_pred[:, i])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Pred 0', 'Pred 1'],
                    yticklabels=['True 0', 'True 1'],
                    cbar=False)
        ax.set_title(label_names[i].title(), fontsize=11, fontweight='bold')
    for ax in axes[n_classes:]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=400, bbox_inches='tight')
    plt.show()
    print(f'Saved → {save_path}')


def print_metrics(model_name, y_true, y_pred, y_score):
    print(f'=== {model_name} ===')
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    print(f'Accuracy (subset)  : {accuracy_score(y_true, y_pred):.4f}')
    print(f'Hamming Loss       : {hamming_loss(y_true, y_pred):.4f}')
    print(f'Coverage Error     : {coverage_error(y_true, y_score):.4f}')
    print(f'Micro F1           : {f1_score(y_true, y_pred, average="micro"):.4f}')
    print(f'Macro F1           : {f1_score(y_true, y_pred, average="macro"):.4f}')
    print(f'Jaccard Macro      : {jaccard_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
    print()


def tune_thresholds(y_true, y_score, model_name):
    """Per-class threshold tuning via precision-recall curve. Returns optimal_thresholds dict."""
    optimal_thresholds = {}
    for i in range(n_classes):
        class_id = mlb.classes_[i]
        precision, recall, thresholds = precision_recall_curve(
            y_true[:, i], y_score[:, i]
        )
        with np.errstate(divide='ignore', invalid='ignore'):
            f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1])
            f1_scores = np.nan_to_num(f1_scores)
        best_idx  = np.argmax(f1_scores)
        best_th   = thresholds[best_idx]
        optimal_thresholds[class_id] = best_th

        f1_default   = f1_score(y_true[:, i], (y_score[:, i] > 0.5).astype(int))
        f1_optimized = f1_score(y_true[:, i], (y_score[:, i] > best_th).astype(int))
        print(f'{label_dict[class_id]:12} | Threshold: {best_th:.3f} | '
              f'F1 default: {f1_default:.3f} | F1 tuned: {f1_optimized:.3f} | '
              f'Delta: {f1_optimized - f1_default:+.3f}')

    return optimal_thresholds


def apply_thresholds(y_score, optimal_thresholds):
    """Apply per-class thresholds to probability scores."""
    y_pred_thresh = np.zeros_like(y_score, dtype=int)
    for i in range(n_classes):
        y_pred_thresh[:, i] = (y_score[:, i] > optimal_thresholds[mlb.classes_[i]]).astype(int)
    return y_pred_thresh


def summary_row(name, y_true, y_pred):
    return {
        'Model':         name,
        'Accuracy':      round(accuracy_score(y_true, y_pred), 4),
        'Hamming Loss':  round(hamming_loss(y_true, y_pred), 4),
        'Micro F1':      round(f1_score(y_true, y_pred, average='micro'), 4),
        'Macro F1':      round(f1_score(y_true, y_pred, average='macro'), 4),
        'Jaccard Macro': round(jaccard_score(y_true, y_pred, average='macro', zero_division=0), 4),
    }

In [ ]:
X_train_sparse = X_train
X_val_sparse   = X_val
X_test_sparse  = X_test

In [ ]:
model_dt_base = OneVsRestClassifier(DecisionTreeClassifier(
    max_depth=20, class_weight='balanced', random_state=42
))
model_dt_base.fit(X_train_sparse, y_train_multilabel)

y_pred_dt_base  = model_dt_base.predict(X_test_sparse)
y_score_dt_base = model_dt_base.predict_proba(X_test_sparse)

print_metrics('Decision Tree — Base', y_test_multilabel, y_pred_dt_base, y_score_dt_base)

plot_roc(y_test_multilabel, y_score_dt_base,
         title='ROC — Decision Tree (Base)', save_path='outputs/plots/roc_dt_base.png')
plot_confusion_matrices(y_test_multilabel, y_pred_dt_base,
                        title='Confusion Matrices — Decision Tree (Base)', save_path='outputs/plots/cm_dt_base.png')

In [ ]:
model_dt_base = OneVsRestClassifier(DecisionTreeClassifier(
    max_depth=20, class_weight='balanced', random_state=42
))
model_dt_base.fit(X_train_sparse, y_train_multilabel)

y_pred_dt_base  = model_dt_base.predict(X_test_sparse)
y_score_dt_base = model_dt_base.predict_proba(X_test_sparse)

print_metrics('Decision Tree — Base', y_test_multilabel, y_pred_dt_base, y_score_dt_base)

plot_roc(y_test_multilabel, y_score_dt_base,
         title='ROC — Decision Tree (Base)', save_path='outputs/plots/roc_dt_base.png')
plot_confusion_matrices(y_test_multilabel, y_pred_dt_base,
                        title='Confusion Matrices — Decision Tree (Base)', save_path='outputs/plots/cm_dt_base.png')

In [ ]:
def dt_objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 5, 40),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'class_weight':      trial.suggest_categorical('class_weight', ['balanced', None]),
        'criterion':         trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'random_state': 42
    }
    clf = OneVsRestClassifier(DecisionTreeClassifier(**params))
    clf.fit(X_train_sparse, y_train_multilabel)
    y_pred_val = clf.predict(X_val_sparse)
    return jaccard_score(y_val_multilabel, y_pred_val, average='macro', zero_division=0)

study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(dt_objective, n_trials=50, show_progress_bar=True)

print('Best params :', study_dt.best_params)
print('Best Jaccard:', study_dt.best_value)best_dt = study_dt.best_params
model_dt_optuna = OneVsRestClassifier(DecisionTreeClassifier(**best_dt, random_state=42))
model_dt_optuna.fit(X_train_sparse, y_train_multilabel)

y_pred_dt_optuna  = model_dt_optuna.predict(X_test_sparse)
y_score_dt_optuna = model_dt_optuna.predict_proba(X_test_sparse)

print_metrics('Decision Tree — Optuna', y_test_multilabel, y_pred_dt_optuna, y_score_dt_optuna)

plot_roc(y_test_multilabel, y_score_dt_optuna,
         title='ROC — Decision Tree (Optuna)', save_path='outputs/plots/roc_dt_optuna.png')
plot_confusion_matrices(y_test_multilabel, y_pred_dt_optuna,
                        title='Confusion Matrices — Decision Tree (Optuna)', save_path='outputs/plots/cm_dt_optuna.png')